# KDD Process Volcano Data Analysis

In [34]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

## Data Cleaning and Preprocessing

In [35]:
def load_data(filepath="volcano-events.tsv"):
    try:
        df = pd.read_csv(filepath, sep='\t')
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return pd.DataFrame()

    df.rename(columns={
        'Damage ($Mil)': 'Damage_Millions',
        'Total Damage ($Mil)': 'Total_Damage_Millions',
        'Elevation (m)': 'Elevation',
        'Total Deaths': 'Total_Deaths',
        'Total Injuries': 'Total_Injuries'
    }, inplace=True)

    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df.dropna(subset=['Year'], inplace=True)
    df['VEI'] = pd.to_numeric(df['VEI'], errors='coerce')
    df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce').fillna(0)
    df['Damage_Millions'] = pd.to_numeric(df['Damage_Millions'], errors='coerce').fillna(0)
    df.dropna(subset=['Latitude', 'Longitude', 'Country'], inplace=True)
    return df

## Spatial Analysis

In [36]:
def get_map_figure(df):
    df_map = df.copy()
    df_map['VEI_Size'] = df_map['VEI'].fillna(0.5)

    fig = px.scatter_geo(
        df_map,
        lat="Latitude",
        lon="Longitude",
        color="Type",
        size="VEI_Size",
        hover_name="Name",
        hover_data={"Country": True, "Year": True, "Deaths": True, "VEI_Size": False},
        title="Global Volcano Distribution",
        projection="natural earth",
        size_max=15,
        template="plotly_dark",
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    fig.update_layout(
        margin={"r":0,"t":50,"l":0,"b":0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Temporal Analysis

In [37]:
def get_frequency_figure(df):
    fig = px.histogram(
        df, 
        x="Year", 
        title="Eruption Frequency",
        nbins=100,
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722')
    fig.update_layout(
        xaxis_title="Year", 
        yaxis_title="Count",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Impact Analysis

In [38]:
def get_impact_figure(df):
    top_deadly = df.nlargest(10, 'Deaths').sort_values('Deaths', ascending=True)
    fig = px.bar(
        top_deadly,
        x="Deaths",
        y="Name",
        orientation='h',
        text="Deaths",
        title="Top 10 Deadliest Eruptions",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722', textposition='outside')
    fig.update_layout(
        xaxis_title="Deaths", 
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Correlation Analysis

In [9]:
df.columns

Index(['Search Parameters', 'Year', 'Mo', 'Dy', 'Tsu', 'Eq', 'Name',
       'Location', 'Country', 'Latitude', 'Longitude', 'Elevation', 'Type',
       'VEI', 'Agent', 'Deaths', 'Death Description', 'Missing',
       'Missing Description', 'Injuries', 'Injuries Description',
       'Damage_Millions', 'Damage Description', 'Houses Destroyed',
       'Houses Destroyed Description', 'Total_Deaths',
       'Total Death Description', 'Total Missing', 'Total Missing Description',
       'Total_Injuries', 'Total Injuries Description', 'Total_Damage_Millions',
       'Total Damage Description', 'Total Houses Destroyed',
       'Total Houses Destroyed Description'],
      dtype='object')

In [12]:
pd.set_option('display.max_columns', None)
df.head()



,Search Parameters,Year,Mo,Dy,Tsu,Eq,Name,Location,Country,Latitude,Longitude,Elevation,Type,VEI,Agent,Deaths,Death Description,Missing,Missing Description,Injuries,Injuries Description,Damage_Millions,Damage Description,Houses Destroyed,Houses Destroyed Description,Total_Deaths,Total Death Description,Total Missing,Total Missing Description,Total_Injuries,Total Injuries Description,Total_Damage_Millions,Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description
1,NaN,-4360.0,NaN,NaN,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,-4350.0,NaN,NaN,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,P,0.0,3.0,NaN,NaN,NaN,NaN,0.0,3.0,NaN,3.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
3,NaN,-4050.0,NaN,NaN,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-4000.0,NaN,NaN,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,T,0.0,1.0,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
5,NaN,-3580.0,NaN,NaN,NaN,NaN,Taal,Luzon-Philippines,Philippines,14.002,120.993,311.0,Stratovolcano,6.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
def get_median_deaths_by_VIE_figure(df):
    # Analysis 1: VEI vs Impact (Median Deaths)
    # Shows the "typical" lethality of different eruption magnitudes
    df_vei = df.groupby('VEI')['Total_Deaths'].median().reset_index()
    fig = px.bar(
        df_vei, x="VEI", y="Total_Deaths",
        title="Median Deaths by VEI",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722')
    fig.update_layout(
        xaxis_title="Volcanic Explosivity Index (VEI)",
        yaxis_title="Median Total Deaths",
        paper_bgcolor='rgba(0,0,0,0)', 
        plot_bgcolor='rgba(0,0,0,0)', 
        font=dict(color="white")
    )
    return fig

def get_top_countries_by_historical_deaths_figure(df):
    # Analysis 2: Geographic Risk (Top 10 Countries by Total Deaths)
    df_country = df.groupby('Country')['Total_Deaths'].sum().nlargest(10).sort_values(ascending=True).reset_index()
    fig = px.bar(
        df_country, x="Total_Deaths", y="Country", orientation='h',
        title="Top 10 Countries by Historical Fatalities",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722')
    fig.update_layout(
        xaxis_title="Total Confirmed Deaths",
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)', 
        plot_bgcolor='rgba(0,0,0,0)', 
        font=dict(color="white")
    )
    return fig

def get_with_and_without_indirect_deaths_by_type_figure(df):
    # Analysis 3: Secondary Hazards Impact
    # Compares average lethality of events with vs. without Tsunami/Earthquake
    res = []
    
    # Tsunami Analysis
    tsu_yes = df[df['Tsu'].notna()]['Total_Deaths'].mean()
    tsu_no = df[df['Tsu'].isna()]['Total_Deaths'].mean()
    res.append({'Hazard': 'Tsunami', 'Status': 'With', 'Avg_Deaths': tsu_yes})
    res.append({'Hazard': 'Tsunami', 'Status': 'Without', 'Avg_Deaths': tsu_no})
    
    # Earthquake Analysis
    eq_yes = df[df['Eq'].notna()]['Total_Deaths'].mean()
    eq_no = df[df['Eq'].isna()]['Total_Deaths'].mean()
    res.append({'Hazard': 'Earthquake', 'Status': 'With', 'Avg_Deaths': eq_yes})
    res.append({'Hazard': 'Earthquake', 'Status': 'Without', 'Avg_Deaths': eq_no})
    
    df_hazards = pd.DataFrame(res)
    
    fig = px.bar(
        df_hazards, x="Hazard", y="Avg_Deaths", color="Status", barmode="group",
        title="Impact Amplification by Secondary Hazards",
        template="plotly_dark",
        color_discrete_map={'With': '#ff5722', 'Without': '#757575'}
    )
    fig.update_layout(
        yaxis_title="Average Deaths per Event",
        paper_bgcolor='rgba(0,0,0,0)', 
        plot_bgcolor='rgba(0,0,0,0)', 
        font=dict(color="white")
    )
    return fig

def get_volcano_type_figure(df):
    # Analysis 4: Volcanic Type vs Impact
    df_type = df.groupby('Type')['Total_Deaths'].sum().reset_index()
    df_type = df_type[df_type['Total_Deaths'] > 0] # Filter out 0 for cleaner chart
    
    fig = px.treemap(
        df_type, path=['Type'], values='Total_Deaths',
        title="Total Deaths by Volcano Type",
        template="plotly_dark",
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', 
        plot_bgcolor='rgba(0,0,0,0)', 
        font=dict(color="white")
    )
    return fig


fig1 = get_median_deaths_by_VIE_figure(df)
fig2 = get_top_countries_by_historical_deaths_figure(df)
fig3 = get_with_and_without_indirect_deaths_by_type_figure(df)
fig4 = get_volcano_type_figure(df)

fig1.show()
fig2.show()
fig3.show()
fig4.show()

## UI for Dashboard

In [ ]:
# Initialize App
app = Dash(__name__)

# Load Data
df = load_data()

# Styles
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#111111",
    "color": "white"
}

CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
    "background-color": "#000000",
    "min-height": "100vh",
    "color": "white"
}

CARD_STYLE = {
    "background-color": "#1e1e1e",
    "padding": "20px",
    "border-radius": "10px",
    "margin-bottom": "20px",
    "box-shadow": "0 4px 6px rgba(0,0,0,0.3)"
}

# Layout
app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Volcano Insights", style={'font-size': '20px', 'margin-bottom': '20px', 'color': '#ff5722'}),
        html.Hr(style={'border-color': '#333'}),
        html.P("Filters", style={'color': '#888'}),
        
        html.Label("Year Range", style={'margin-top': '20px'}),
        dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            value=[df['Year'].min(), df['Year'].max()],
            marks={str(year): str(year) for year in range(int(df['Year'].min()), int(df['Year'].max()), 1000)},
            tooltip={"placement": "bottom", "always_visible": True},
            className="dark-slider"
        ),
        
        html.Label("Country", style={'margin-top': '20px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country'].unique())],
            placeholder="All Countries",
            style={'color': 'black'} # Dropdown text needs to be black to be visible on white bg of default dropdown
        )
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        html.H1("Volcano Insights Dashboard", style={'margin-bottom': '5px'}),
        html.P("Analyzing Significant Volcanic Eruptions", style={'color': '#888', 'margin-bottom': '30px'}),

        # KPI Row
        html.Div([
            html.Div([
                html.H4("Total Eruptions", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-eruptions', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Deaths", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-deaths', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Damage ($M)", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-damage', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

        # Charts Row 1
        html.Div([
            html.Div([dcc.Graph(id='map-graph')], style={**CARD_STYLE, 'flex': '2', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='time-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'margin-bottom': '20px'}),

        # Charts Row 2 - impact charts
        html.Div([
            html.Div([dcc.Graph(id='impact-graph')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
        ], style={'display': 'flex'}),

        # Correlation charts
        # Correlation chart row 1
        html.Div([
            html.Div([dcc.Graph(id='median_deaths_by_VIE')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='top_countries_by_historical_deaths')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex'}),
        
        # Correlation chart row 2
        html.Div([
            html.Div([dcc.Graph(id='with_and_without_indirect_deaths_by_type')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'})
        ], style={'display': 'flex'})

    ], style=CONTENT_STYLE)
],style={'backgroundColor': '#000000', 'minHeight': '100vh'})

# Callbacks
@app.callback(
    [Output('map-graph', 'figure'),
     Output('time-graph', 'figure'),
     Output('impact-graph', 'figure'),
     Output('median_deaths_by_VIE', 'figure'),
     Output('top_countries_by_historical_deaths', 'figure'),
     Output('with_and_without_indirect_deaths_by_type', 'figure'),
     Output('volcano_type', 'figure'),
     Output('kpi-eruptions', 'children'),
     Output('kpi-deaths', 'children'),
     Output('kpi-damage', 'children')],
    [Input('country-dropdown', 'value'),
     Input('year-slider', 'value')]
)
def update_dashboard(selected_country, year_range):
    # Filter Data
    dff = df.copy()
    if selected_country:
        dff = dff[dff['Country'] == selected_country]
    
    if year_range:
        dff = dff[(dff['Year'] >= year_range[0]) & (dff['Year'] <= year_range[1])]

    if dff.empty:
        dff = df # Fallback if empty

    # KPIs
    total_eruptions = len(dff)
    total_deaths = f"{int(dff['Deaths'].sum()):,}"
    total_damage = f"${dff['Damage_Millions'].sum():,.0f}"
    # Recommended Updates for EVERY figure (e.g., inside get_map_figure)
    fig.update_layout(
        # 1. Set the chart background to match the card color
        plot_bgcolor='#1e1e1e', 
        # 2. Set the background of the entire figure area (paper) to match
        paper_bgcolor='#1e1e1e', 
        # 3. Set all text (titles, labels) to white
        font=dict(color='white'),
        # 4. Remove the default border/outline padding around the charts
        margin=dict(l=10, r=10, t=40, b=10) 
    )


    # Figures
    fig1 = get_map_figure(dff)
    fig2 = get_frequency_figure(dff)
    fig3 = get_impact_figure(dff)
    fig_median_deaths_by_VIE = get_median_deaths_by_VIE(dff)
    fig_top_countries_by_historical_deaths = get_top_countries_by_historical_deaths_figure(df)
    fig_with_and_without_indirect_deaths_by_type = get_with_and_without_indirect_deaths_by_type_figure(df)
    fig_volcano_type = get_volcano_type_figure(df)
    

    return fig1, fig2, fig3, fig_median_deaths_by_VIE, fig_top_countries_by_historical_deaths, fig_with_and_without_indirect_deaths_by_type, fig_volcano_type, total_eruptions, total_deaths, total_damage

if __name__ == '__main__':
    print("Launching Dashboard...")
    print("Dashboard launched at: http://127.0.0.1:7860")
    app.run(host='127.0.0.1', port=7860, debug=True)

Launching Dashboard...
Dashboard launched at: http://127.0.0.1:7860
---------------------------------------------------------------------------
DuplicateIdError                          Traceback (most recent call last)
DuplicateIdError: Duplicate component id found in the initial layout: `corr-graph`



---------------------------------------------------------------------------
SchemaLengthValidationError               Traceback (most recent call last)
SchemaLengthValidationError: Schema: [<Output `map-graph.figure`>, <Output `time-graph.figure`>, <Output `impact-graph.figure`>, <Output `corr-graph.figure`>, <Output `kpi-eruptions.children`>, <Output `kpi-deaths.children`>, <Output `kpi-damage.children`>]
                Path: ()
                Expected length: 7
                Received value of length 10:
                    [Figure({
    'data': [{'customdata': array([['New Zealand', -4360.0, 0.0, 6.0],
                                   ['Japan', -4350.0, 0.0, 7.0],
                                   ['Nicaragua', -4050.0, 0.0, 6.0],
                                   ...,
                                   ['Japan', 2016.0, 0.0, 0.5],
                                   ['Italy', 2017.0, 3.0, 0.5],
                                   ['Indonesia', 2018.0, 0.0, 3.0]], shape=(69, 4)